# Notebook 03 – Retrieval Experiments (Part B)

1. Show vector-only baseline
2. Show BM25 baseline
3. Show hybrid (RRF) fusion — the winner
4. **Document ≥3 failure cases** where vector-only fails and hybrid fixes it

In [1]:
%pip install -q pandas pypdf numpy sentence-transformers openai tiktoken python-dotenv scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
sys.path.insert(0, os.path.join('..', 'backend'))
from pathlib import Path
from raghana.vectorstore import VectorStore
from raghana.bm25 import BM25
from raghana.retrieve import retrieve

ARTIFACTS = Path('../backend/artifacts')
vs = VectorStore.load(ARTIFACTS / 'embeddings')
bm25 = BM25.load(ARTIFACTS / 'bm25.pkl')
print(f'Loaded {len(vs)} chunks')

Loaded 338 chunks


In [3]:
def show_results(query, mode='hybrid', k=5):
    results = retrieve(query, vs, bm25, k=k, mode=mode)
    print(f'\n=== [{mode.upper()}] "{query}" ===')
    for r in results:
        print(f'  [{r["chunk_id"]}] score={r.get("score",0):.4f}  src={r["source"]}  {r["text"][:80].strip()!r}')
    return results

## 3.1 Standard queries — all modes

In [4]:
query = 'NPP votes in Ashanti Region 2020'
for mode in ['vector', 'bm25', 'hybrid']:
    show_results(query, mode=mode)


=== [VECTOR] "NPP votes in Ashanti Region 2020" ===
  [csv_0040] score=0.6698  src=csv  'In the 2008 presidential election in Ashanti\xa0Region:\n  Nana Addo (NPP) received'
  [csv_0020] score=0.6586  src=csv  'In the 2000 presidential election in Ashanti\xa0Region:\n  J. A. Kuffour (NPP) recei'
  [csv_0010] score=0.6526  src=csv  'In the 1996 presidential election in Ashanti\xa0Region:\n  J. A. Kuffour (NPP) recei'
  [csv_0030] score=0.6399  src=csv  'In the 2004 presidential election in Ashanti\xa0Region:\n  J. A. Kuffour (NPP) recei'
  [csv_0083] score=0.6380  src=csv  'In the 2020 presidential election in Ashanti\xa0Region:\n  Nana Akufo Addo (NPP) rec'

=== [BM25] "NPP votes in Ashanti Region 2020" ===
  [csv_0083] score=14.6032  src=csv  'In the 2020 presidential election in Ashanti\xa0Region:\n  Nana Akufo Addo (NPP) rec'
  [csv_0000] score=12.3086  src=csv  'In the 1992 presidential election in Ashanti Region:\n  Albert Adu Boahen (NPP) r'
  [csv_0010] score=12.2884  src=csv  

## 3.2 Failure Case 1 — Year specificity

**Query:** `CPP votes in Volta Region 2016`

**Expected failure:** vector-only returns 2020 Volta chunks because the semantic
content is similar. BM25 anchors on `2016` + `CPP` + `Volta`.

In [5]:
failure1 = 'CPP votes in Volta Region 2016'
v_results = show_results(failure1, mode='vector')
h_results = show_results(failure1, mode='hybrid')

# Check if the correct year/region chunk is in each result set
def contains_gold(results, year, region, party):
    for r in results:
        m = r.get('metadata', {})
        if m.get('year') == year and region.lower() in m.get('region', '').lower():
            if party.lower() in r.get('text', '').lower():
                return True
    return False

print('\nVector finds correct chunk:', contains_gold(v_results, 2016, 'Volta', 'CPP'))
print('Hybrid finds correct chunk:', contains_gold(h_results, 2016, 'Volta', 'CPP'))


=== [VECTOR] "CPP votes in Volta Region 2016" ===
  [csv_0079] score=0.6434  src=csv  'In the 2016 presidential election in Volta\xa0Region:\n  Nana Akufo Addo (NPP) recei'
  [csv_0095] score=0.6005  src=csv  'In the 2020 presidential election in Volta\xa0Region:\n  Nana Akufo Addo (NPP) recei'
  [csv_0008] score=0.5877  src=csv  'In the 1992 presidential election in Volta Region:\n  Albert Adu Boahen (NPP) rec'
  [csv_0028] score=0.5792  src=csv  'In the 2000 presidential election in Volta\xa0Region:\n  J. A. Kuffour (NPP) receive'
  [csv_0063] score=0.5715  src=csv  'In the 2012 presidential election in Volta\xa0Region:\n  Nana Akufo Addo (NPP) recei'

=== [HYBRID] "CPP votes in Volta Region 2016" ===
  [csv_0079] score=0.0328  src=csv  'In the 2016 presidential election in Volta\xa0Region:\n  Nana Akufo Addo (NPP) recei'
  [csv_0028] score=0.0315  src=csv  'In the 2000 presidential election in Volta\xa0Region:\n  J. A. Kuffour (NPP) receive'
  [csv_0095] score=0.0313  src=csv  'In 

## 3.3 Failure Case 2 — Exact numeric phrase

**Query:** `debt-to-GDP target`

**Expected failure:** very short query, vector retrieval may surface semantically
related but incorrect budget passages. BM25 matches the exact phrase.

In [6]:
failure2 = 'debt-to-GDP target'
show_results(failure2, mode='vector')
show_results(failure2, mode='hybrid')


=== [VECTOR] "debt-to-GDP target" ===
  [pdf_para_0018] score=0.5960  src=pdf  '15 75. Moreover, the forthcoming debt service of both Domestic and Eurobond debt'
  [pdf_para_0069] score=0.5721  src=pdf  '66 349. Mr. Speaker, the primary expenditure is projected to rise modestly to 15'
  [pdf_para_0032] score=0.5696  src=pdf  '29 \n2024 Budget Balances & Financing Operations \n140. Mr. Speaker, Government ’s'
  [pdf_para_0041] score=0.5351  src=pdf  '38 182. Mr. Speaker, the DSA assessment classified Ghana’s external and public d'
  [pdf_para_0019] score=0.5337  src=pdf  '16 83. Mr. Speaker, it seems the debt restructuring undertaken by the previous a'

=== [HYBRID] "debt-to-GDP target" ===
  [pdf_para_0032] score=0.0320  src=pdf  '29 \n2024 Budget Balances & Financing Operations \n140. Mr. Speaker, Government ’s'
  [pdf_para_0041] score=0.0306  src=pdf  '38 182. Mr. Speaker, the DSA assessment classified Ghana’s external and public d'
  [pdf_para_0018] score=0.0299  src=pdf  '15 75. M

[{'chunk_id': 'pdf_para_0032',
  'source': 'pdf',
  'text': '29 \n2024 Budget Balances & Financing Operations \n140. Mr. Speaker, Government ’s fiscal operations for 2024 resulted in an overall cash deficit \nof GH¢6 1,411 million ( 5.2% of GDP), against the target of GH¢54,142 million (5.3% \nof GDP). The cash deficit was financed from both domestic and external sources. The \ncorresponding Primary Balance (on cash basis) was a deficit of GH¢1 4,618 million \n(1.2% of GDP), against the target deficit of GH¢6,144 million (0.6% of GDP). \n \n141. Net domestic financing amounted to GH¢ 41,537 million ( 3.5% of GDP), against the \ntarget of GH¢ 40,195 million ( 3.9% of GDP) and accounted for 6 7.6 percent of total \nfinancing. Of the total NDF, the Non -Bank sector financed GH¢48,194 million while \nfinancing from commercial banks was GH¢1,063 million. Bank of Ghana financing was \na net buildup of GH¢7,720 million. \n \n142. Mr. Speaker, Foreign Financing (Net) amounted to GH¢2 1,817 mil

## 3.4 Failure Case 3 — Paraphrase with proper nouns

**Query:** `parliamentary seats won by NPP in Ashanti`

**Expected failure:** parliamentary seats aren't in the CSV (it's presidential data),
so all retrieval methods should surface relevant passages or CSV proxies.
Document the actual behaviour — expected or unexpected.

In [7]:
failure3 = 'parliamentary seats won by NPP in Ashanti'
show_results(failure3, mode='vector')
show_results(failure3, mode='hybrid')
# Observe: hybrid should surface Ashanti/NPP CSV chunks even though 'parliamentary' is absent


=== [VECTOR] "parliamentary seats won by NPP in Ashanti" ===
  [csv_0040] score=0.6825  src=csv  'In the 2008 presidential election in Ashanti\xa0Region:\n  Nana Addo (NPP) received'
  [csv_0010] score=0.6522  src=csv  'In the 1996 presidential election in Ashanti\xa0Region:\n  J. A. Kuffour (NPP) recei'
  [csv_0030] score=0.6499  src=csv  'In the 2004 presidential election in Ashanti\xa0Region:\n  J. A. Kuffour (NPP) recei'
  [csv_0051] score=0.6472  src=csv  'In the 2012 presidential election in Ashanti\xa0Region:\n  Nana Akufo Addo (NPP) rec'
  [csv_0067] score=0.6409  src=csv  'In the 2016 presidential election in Ashanti\xa0Region:\n  Nana Akufo Addo (NPP) rec'

=== [HYBRID] "parliamentary seats won by NPP in Ashanti" ===
  [csv_0010] score=0.0320  src=csv  'In the 1996 presidential election in Ashanti\xa0Region:\n  J. A. Kuffour (NPP) recei'
  [csv_0040] score=0.0313  src=csv  'In the 2008 presidential election in Ashanti\xa0Region:\n  Nana Addo (NPP) received'
  [csv_0030] scor

[{'chunk_id': 'csv_0010',
  'source': 'csv',
  'text': 'In the 1996 presidential election in Ashanti\xa0Region:\n  J. A. Kuffour (NPP) received 827,804 votes (65.80%)\n  J. J. Rawlings (NDC) received 412,474 votes (32.79%)\n  Edward Mahama (PNC) received 17,736 votes (1.41%)',
  'metadata': {'year': 1996, 'region': 'Ashanti\xa0Region', 'row_count': 3},
  'score': 0.03200204813108039,
  'vector_score': 0.6521940231323242,
  'bm25_score': 7.892162451778647,
  'rrf_score': 0.03200204813108039,
  'rank': 1},
 {'chunk_id': 'csv_0040',
  'source': 'csv',
  'text': 'In the 2008 presidential election in Ashanti\xa0Region:\n  Nana Addo (NPP) received 1,214,350 votes (72.40%)\n  J. A Mills (NDC) received 438,234 votes (26.13%)\n  Papa K. Nduom (CPP) received 11,937 votes (0.71%)\n  Edward Mahama (PNC) received 5,464 votes (0.33%)\n  Kwesi Amoafo-Yeboah (IND) received 2,517 votes (0.15%)',
  'metadata': {'year': 2008, 'region': 'Ashanti\xa0Region', 'row_count': 5},
  'score': 0.03131881575727918,

## 3.5 Summary table

Fill in the results after running the cells above.

| # | Query | Vector finds gold? | Hybrid finds gold? | Why vector fails |
|---|---|---|---|---|
| 1 | CPP in Volta 2016 | ? | ? | Year tokens lost in dense space |
| 2 | debt-to-GDP target | ? | ? | Short exact phrase, low semantic signal |
| 3 | parliamentary NPP Ashanti | ? | ? | Paraphrase drift |

**Manual log:** record in `experiment_logs/partB_retrieval.md`